# Figure G02: how much of the entropy–length trend is plug-in estimator bias (GTDB r232, genome_entropy #101)

**Status:** draft, Gadi. The ID is provisional. The companion calibration figure is GS02.

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figG02_mean_curves.tsv.gz` | Gadi `entropy_null/empirical/length_entropy_{bac,arc}.tsv` (matched stratum 3, 1 aa bins) and `entropy_null/entropy_null_quantiles.tsv` (i.i.d. mean, interpolated to each length) | `33_empirical_length_null.pbs` (full population), `31_entropy_null.pbs` (400,000 multinomial draws per length) |

- **Source:** `linsalrob/genome_entropy` `main` at `97a792f`.
- **Reduction:** `manuscript/prep/gtdb_prepare_data.py --only G02`. It uses the same selection and interpolation as `38_bias_vs_biology.py`. The 30 rows of the published `bias_decomposition.tsv` are reproduced exactly from this table, and the check cell below prints the length-300 rows.
- **Definitions:**
  - The **observed** curve is the mean entropy of ORFs matched to a deposited CDS, at each length from 90 to 999 aa (1 aa bins). It covers 301,622,163 bacterial and 9,331,047 archaeal matched ORFs, which is every matched ORF shorter than 1,000 aa.
  - The **estimator bias** curve is the mean plug-in entropy of i.i.d. sequences of the same length, drawn from the domain's pooled residue composition (21 protein symbols, 20 3Di states, 12 states). It contains no biology, only the downward bias of the plug-in estimator at finite length.
  - Both curves are shown relative to L = 100 aa.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}

import math
from matplotlib.colors import LogNorm

CLASS = {"matched": "#0072B2", "unmatched": "#E69F00"}
CEILINGS = [(0.0, "log$_2$1"), (1.0, "log$_2$2"), (math.log2(3), "log$_2$3"), (2.0, "log$_2$4")]


def ceilings(ax, axis="y", label=False):
    for v, t in CEILINGS:
        (ax.axhline if axis == "y" else ax.axvline)(
            v, color=MUTED, lw=0.6, ls=(0, (1, 2)), zorder=3)
        if label:
            ax.text(v, 1.0, t, transform=ax.get_xaxis_transform(), rotation=90,
                    ha="right", va="top", fontsize=5.8, color=MUTED)


In [ ]:
cur = pd.read_csv(DATA / "figG02_mean_curves.tsv.gz", sep="\t")
ALPH = [("protein", "protein"), ("three_di", "3Di"), ("twelve_state", "12-state")]
OBS, BIAS = "#0072B2", "#D55E00"
fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.4), sharex=True)
for i, dom in enumerate(("bacteria", "archaea")):
    for j, (a, lab) in enumerate(ALPH):
        ax = axes[i, j]
        c = cur[(cur.domain == dom) & (cur.alphabet == a)].set_index("aa_length")
        o = c.observed_mean - c.observed_mean[100]
        b = c.bias_mean - c.bias_mean[100]
        ax.fill_between(c.index, b, o, where=o >= b, color=OBS, alpha=0.18, lw=0,
                        label="rise not explained by bias")
        ax.plot(c.index, o, color=OBS, lw=1.1, label="observed (ORFs matching a GenBank CDS)")
        ax.plot(c.index, b, color=BIAS, lw=1.1, ls="--", label="expected rise from estimator bias alone (random sequences)")
        ax.axhline(0, color=MUTED, lw=0.5)
        ax.set_xscale("log")
        ax.set_xlim(90, 999)
        ax.set_xticks([100, 200, 300, 500, 999], ["100", "200", "300", "500", "999"])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        o3, b3 = o[300], b[300]
        xy = (0.97, 0.05, "right", "bottom") if a == "protein" else (0.03, 0.97, "left", "top")
        ax.text(xy[0], xy[1], f"100→300 aa\nobserved {o3:+.2f}\nbias {b3:+.2f} bits",
                transform=ax.transAxes, ha=xy[2], va=xy[3], fontsize=6, color=INK)
        if i == 0:
            ax.set_title(f"{lab} entropy", pad=22)
        else:
            ax.set_xlabel("ORF length (aa)")
        if j == 0:
            ax.set_ylabel("change in mean entropy\nsince 100 aa (bits)")
            panel(ax, "ab"[i])
            ax.text(0.02, 1.04, dom.capitalize(), transform=ax.transAxes, fontsize=8,
                    fontweight="bold", ha="left", va="bottom")
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.06), fontsize=6.5)
fig.tight_layout(rect=(0, 0.04, 1, 1))
save(fig, "figG02_estimator_bias")


In [ ]:
rows = []
for (d, a), c in cur.groupby(["domain", "alphabet"]):
    c = c.set_index("aa_length")
    for L in (150, 200, 300, 500, 800):
        o = c.observed_mean[L] - c.observed_mean[100]
        b = c.bias_mean[L] - c.bias_mean[100]
        rows.append(dict(domain=d, alphabet=a, to_length=L, observed_rise=round(o, 4),
                         bias_rise=round(b, 4), artefact_pct=round(100 * b / o, 1)))
t = pd.DataFrame(rows)
print(t[t.to_length == 300].to_string(index=False))
print("matched ORFs in the curves:", cur.groupby("domain").apply(lambda g: g[g.alphabet == "protein"].n.sum()).to_dict())


## Caption (draft)

**Figure G02. The rise of 3Di entropy with ORF length is mostly biological; the rise of protein entropy is plug-in estimator bias.**

The mean entropy of all GTDB r232 ORFs matched to a deposited CDS is shown at each length from 90 to 999 aa (solid), relative to its value at 100 aa. Alongside it is the expected rise for i.i.d. sequences of the same length and pooled composition (dashed), which is the downward bias of the plug-in estimator and nothing else. Shading marks the part of the observed rise that the bias does not explain. **(a)** Bacteria, 301.6 M ORFs. **(b)** Archaea, 9.33 M ORFs.

Between 100 and 300 aa:
- **Protein entropy:** the bias exceeds the observed rise (bias +0.11 bits against observed +0.08 bacteria / +0.09 archaea), so the apparent protein-entropy trend is entirely an artefact.
- **3Di entropy:** observed +0.88 / +0.76 bits, of which only +0.11 / +0.10 is bias (12% / 14%). Longer proteins genuinely use more distinct structural states.
- **12-state:** observed +0.85 / +0.75 bits, of which +0.06 is bias (7–8%).

Length-conditioned comparisons should therefore use the empirical reference (calibration in Figure GS02). Raw protein entropies should not be compared across lengths.
